<!--<badge>--><a href="https://colab.research.google.com/github/JoeChen322/Fintech/blob/main/recommend-system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><!--</badge>-->


In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import gdown
import tqdm

%config InlineBackend.figure_format = 'retina'

In [2]:
url = "https://drive.google.com/uc?id=1TiR645keulkG4pONLpVn-bO4elNBtNpX"
file_path = "Dataset2_Needs.xls"
gdown.download(url, file_path, quiet=False)
needs_df = pd.read_excel(file_path, sheet_name="Needs")
products_df = pd.read_excel(file_path, sheet_name="Products")
metadata_df = pd.read_excel(file_path, sheet_name="Metadata")

print("Needs shape:", needs_df.shape)
print("Products shape:", products_df.shape)
print("Metadata shape:", metadata_df.shape)

Downloading...
From: https://drive.google.com/uc?id=1TiR645keulkG4pONLpVn-bO4elNBtNpX
To: /Users/pengrao/Workspace/Fintech/Dataset2_Needs.xls
100%|██████████| 428k/428k [00:00<00:00, 12.3MB/s]


Needs shape: (5000, 10)
Products shape: (11, 3)
Metadata shape: (29, 2)


In [3]:
needs_df.head()

,ID,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income,Wealth,IncomeInvestment,AccumulationInvestment
0,1,60,0,2,0.228685,0.233355,68.181525,53.260067,0,1
1,2,78,0,2,0.358916,0.170911,21.807595,135.550048,1,0
2,3,33,1,2,0.317515,0.249703,23.252747,66.303678,0,1
3,4,69,1,4,0.767685,0.654597,166.189034,404.997689,1,1
4,5,58,0,3,0.429719,0.349039,21.186723,58.911930,0,0


In [4]:
products_df.head()

,IDProduct,Type,Risk
0,1,1,0.55
1,2,0,0.30
2,3,0,0.12
3,4,0,0.44
4,5,1,0.41


In [5]:
metadata_df.head()

,Metadata,Unnamed: 1
0,Clients,NaN
1,ID,Numerical ID
2,Age,"Age, in years"
3,Gender,"Gender (Female = 1, Male = 0)"
4,FamilyMembers,Number of components


## Data processing

This cell cleans the tabular data, builds the engineered features, and prepares the train/test split used by the recommendation models.


In [6]:
needs_df = needs_df.copy()
products_df = products_df.copy()
metadata_df = metadata_df.copy()

In [7]:
needs_df

,ID,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income,Wealth,IncomeInvestment,AccumulationInvestment
0,1,60,0,2,0.228685,0.233355,68.181525,53.260067,0,1
1,2,78,0,2,0.358916,0.170911,21.807595,135.550048,1,0
2,3,33,1,2,0.317515,0.249703,23.252747,66.303678,0,1
3,4,69,1,4,0.767685,0.654597,166.189034,404.997689,1,1
4,5,58,0,3,0.429719,0.349039,21.186723,58.911930,0,0
...,...,...,...,...,...,...,...,...,...,...
4995,4996,60,1,3,0.609411,0.588353,10.253281,57.368572,0,0
4996,4997,65,1,3,0.523238,0.343272,104.155427,156.824602,1,1
4997,4998,56,1,3,0.433826,0.402771,56.097301,63.283774,0,1
4998,4999,51,1,3,0.559793,0.431419,62.523298,95.357528,0,0


In [8]:
needs_df.columns

Index(['ID', 'Age', 'Gender', 'FamilyMembers', 'FinancialEducation',
       'RiskPropensity', 'Income', 'Wealth', 'IncomeInvestment',
       'AccumulationInvestment'],
      dtype='str')

In [9]:
if "ID" in needs_df.columns:
    needs_df = needs_df.drop(columns=["ID"])

needs_df["Income_log"] = np.log1p(needs_df["Income"].clip(lower=0))
needs_df["Wealth_log"] = np.log1p(needs_df["Wealth"].clip(lower=0))
needs_df["Income_Wealth_Ratio"] = needs_df["Income"] / needs_df["Wealth"].replace(
    0, np.nan
)
needs_df["Income_Wealth_Ratio"] = needs_df["Income_Wealth_Ratio"].fillna(0)

feature_columns = [
    "Age",
    "Gender",
    "FamilyMembers",
    "FinancialEducation",
    "RiskPropensity",
    "Income_log",
    "Wealth_log",
    "Income_Wealth_Ratio",
]
X_base = needs_df[feature_columns].copy()
scaler = MinMaxScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(X_base), columns=feature_columns, index=needs_df.index
)
X_engineered = X_base.copy()

X_train, X_test, y_train_accum, y_test_accum = train_test_split(
    X_base,
    needs_df["AccumulationInvestment"],
    test_size=0.2,
    random_state=42,
    stratify=needs_df["AccumulationInvestment"],
)
_, _, y_train_income, y_test_income = train_test_split(
    X_base,
    needs_df["IncomeInvestment"],
    test_size=0.2,
    random_state=42,
    stratify=needs_df["IncomeInvestment"],
)
test_clients = needs_df.loc[X_test.index].copy()

print("Processed features:", X_base.shape)
print("Train set:", X_train.shape)
print("Test set:", X_test.shape)
X_base.head()

Processed features: (5000, 8)
Train set: (4000, 8)
Test set: (1000, 8)


,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income_log,Wealth_log,Income_Wealth_Ratio
0,0.531646,0.0,0.25,0.222172,0.243105,0.664782,0.468132,0.006922
1,0.759494,0.0,0.25,0.372410,0.170321,0.441614,0.600160,0.000789
2,0.189873,1.0,0.25,0.324649,0.262161,0.453970,0.498951,0.001829
3,0.645570,1.0,0.75,0.843975,0.734110,0.842246,0.756044,0.002156
4,0.506329,0.0,0.50,0.454090,0.377948,0.436064,0.482307,0.001878


In [10]:
accumulation_products = products_df[products_df["Type"] == 1].copy()
income_products = products_df[products_df["Type"] == 0].copy()
test_clients = needs_df.loc[X_test.index].copy()


In [11]:
svm_accum = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42,
)
svm_income = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42,
)

In [12]:
svm_accum.fit(X_train, y_train_accum)
accum_pred = pd.Series(svm_accum.predict(X_test).astype(int), index=X_test.index)
print("Accumulation positive predictions:", int(accum_pred.sum()))

svm_income.fit(X_train, y_train_income)
income_pred = pd.Series(svm_income.predict(X_test).astype(int), index=X_test.index)
print("Income positive predictions:", int(income_pred.sum()))

Accumulation positive predictions: 531
Income positive predictions: 261


In [13]:
age_min = needs_df["Age"].min()
age_max = needs_df["Age"].max()
wealth_min = needs_df["Wealth"].min()
wealth_max = needs_df["Wealth"].max()

# Recommendation Systems

This section adds SVD and autoencoder recommenders built from a client-product interaction matrix.


## SVD


SVD (Singular Value Decomposition) recommends items by factorizing the user-item interaction matrix $R$ into low-dimensional latent factors that capture hidden preferences and item characteristics.

In practice, we approximate:

$$
R \approx U_k \Sigma_k V_k^T
$$

where $k$ is the number of retained latent dimensions.

The predicted preference score for user $u$ and item $i$ is reconstructed from these latent factors (dot product in latent space). Higher reconstructed scores indicate stronger recommendation relevance.

Why this works: SVD compresses noisy sparse interactions into a compact structure, keeping the strongest collaborative signals while removing less informative variance.

This algorithm includes the following steps:

1. Construct user-product interaction matrix $R$
2. Using Truncated SVD for dimensionality reduction
3. Refactor the scoring matrix and generate recommendations
4. Integrated business rules


#### The user-item matrix


We use model-based collaborative filtering algorithm with predicted probability matrix.

This method leverages the **output probabilities** of the already-trained supervised classifiers to construct a dense user–item interaction matrix. Instead of relying on manually crafted scoring rules, we treat the model's predicted propensity for each investment need as a **data-driven confidence score**. This matrix then serves as input to a collaborative filtering algorithm (Truncated SVD) for product recommendation.

Let:

- $P_{\text{acc}}(i) = \mathbb{P}(\text{AccumulationInvestment}=1 \mid \text{client } i)$
- $P_{\text{inc}}(i) = \mathbb{P}(\text{IncomeInvestment}=1 \mid \text{client } i)$

For each client $i$ and product $j$ with product type $\text{Type}(j) \in \{0,1\}$ (0 = Income, 1 = Accumulation), the matrix entry $R_{ij}$ is defined as:

$$
R_{ij} =
\begin{cases}
P_{\text{acc}}(i) \cdot \big(1 - \alpha \cdot |\,\text{RiskPropensity}(i) - \text{Risk}(j)\,|\big), & \text{if Type}(j) = 1 \\[6pt]
P_{\text{inc}}(i) \cdot \big(1 - \alpha \cdot |\,\text{RiskPropensity}(i) - \text{Risk}(j)\,|\big), & \text{if Type}(j) = 0
\end{cases}
$$

where $\alpha \in [0,1]$ is an optional decay factor that penalises risk misalignment. Setting $\alpha = 0$ reduces the matrix to pure predicted probabilities.


In [14]:
# define decay factor for risk penalty
alpha = 0.35

all_clients = needs_df.copy()
all_features = X_base.copy()

accum_proba = pd.Series(
    svm_accum.predict_proba(all_features)[:, 1], index=all_features.index, name="P_acc"
)
income_proba = pd.Series(
    svm_income.predict_proba(all_features)[:, 1], index=all_features.index, name="P_inc"
)

product_meta = products_df.copy()
product_ids = product_meta["IDProduct"].astype(int).tolist()

interaction_matrix = pd.DataFrame(
    index=all_features.index, columns=product_ids, dtype=float
)


# calculate risk penalty and final scores for each product
for _, product in product_meta.iterrows():
    pid = int(product["IDProduct"])
    product_type = int(product["Type"])
    risk = float(product["Risk"])
    risk_penalty = (1.0 - alpha * np.abs(all_clients["RiskPropensity"] - risk)).clip(
        lower=0.0
    )
    base_score = accum_proba if product_type == 1 else income_proba
    interaction_matrix[pid] = (base_score * risk_penalty).astype(float)

n_components = max(
    2, min(6, interaction_matrix.shape[1] - 1, interaction_matrix.shape[0] - 1)
)

In [15]:
# Build a holdout split at interaction-entry level for recommender validation
def build_entry_holdout(matrix_df, test_size=0.2, random_state=42):
    values = matrix_df.values.astype(float)
    candidate_pairs = np.argwhere(values > 0)

    if len(candidate_pairs) < 10:
        raise ValueError(
            "Not enough non-zero entries to create a stable holdout split."
        )

    train_ids, test_ids = train_test_split(
        np.arange(len(candidate_pairs)),
        test_size=test_size,
        random_state=random_state,
    )

    train_values = values.copy()
    holdout_pairs = candidate_pairs[test_ids]
    train_values[holdout_pairs[:, 0], holdout_pairs[:, 1]] = 0.0

    train_df = pd.DataFrame(
        train_values, index=matrix_df.index, columns=matrix_df.columns
    )
    return train_df, holdout_pairs


def apply_business_rules(score_df, client_df, product_df):
    adjusted = score_df.copy()
    wealth_median = client_df["Wealth"].median()

    for _, product in product_df.iterrows():
        pid = int(product["IDProduct"])
        product_type = int(product["Type"])
        product_risk = float(product["Risk"])

        rule_multiplier = pd.Series(1.0, index=adjusted.index)

        if product_type == 1:
            rule_multiplier *= np.where(client_df["Age"] > 70, 0.85, 1.0)
        else:
            rule_multiplier *= np.where(client_df["Age"] < 30, 0.90, 1.0)

        rule_multiplier *= np.where(
            (client_df["Wealth"] < wealth_median) & (product_risk > 0.60),
            0.80,
            1.0,
        )
        rule_multiplier *= np.where(
            np.abs(client_df["RiskPropensity"] - product_risk) > 0.45,
            0.75,
            1.0,
        )
        adjusted[pid] = (adjusted[pid] * rule_multiplier).clip(lower=0.0)

    return adjusted


def evaluate_recommender(score_df, truth_df, holdout_pairs, k=3):
    pred_values = score_df.values.astype(float)
    truth_values = truth_df.values.astype(float)

    rows = holdout_pairs[:, 0]
    cols = holdout_pairs[:, 1]
    rmse = float(
        np.sqrt(np.mean((pred_values[rows, cols] - truth_values[rows, cols]) ** 2))
    )

    user_to_items = {}
    for r, c in holdout_pairs:
        user_to_items.setdefault(int(r), []).append(int(c))

    precision_at_k = []
    recall_at_k = []
    ndcg_at_k = []

    for user_idx, item_indices in user_to_items.items():
        if not item_indices:
            continue

        k_eff = min(k, len(item_indices))
        truth_scores = truth_values[user_idx, item_indices]
        pred_scores = pred_values[user_idx, item_indices]

        true_order = np.argsort(truth_scores)[::-1][:k_eff]
        pred_order = np.argsort(pred_scores)[::-1][:k_eff]

        true_top = {item_indices[i] for i in true_order}
        pred_top = [item_indices[i] for i in pred_order]
        hits = len(set(pred_top).intersection(true_top))

        precision_at_k.append(hits / k_eff)
        recall_at_k.append(hits / len(true_top))

        rel_at_pred = truth_scores[pred_order]
        discounts = np.log2(np.arange(2, k_eff + 2))
        dcg = np.sum((2**rel_at_pred - 1) / discounts)

        ideal_rel = np.sort(truth_scores)[::-1][:k_eff]
        ideal_dcg = np.sum((2**ideal_rel - 1) / discounts)
        ndcg_at_k.append(float(dcg / ideal_dcg) if ideal_dcg > 0 else np.nan)

    return {
        "RMSE": rmse,
        f"Precision@{k}": float(np.nanmean(precision_at_k)),
        f"Recall@{k}": float(np.nanmean(recall_at_k)),
        f"NDCG@{k}": float(np.nanmean(ndcg_at_k)),
    }


interaction_train, holdout_pairs = build_entry_holdout(
    interaction_matrix, test_size=0.2, random_state=42
)
print("Validation holdout pairs:", len(holdout_pairs))

Validation holdout pairs: 11000


### Truncated SVD


In [16]:
svd = TruncatedSVD(n_components=n_components, random_state=42)
latent_users = svd.fit_transform(interaction_matrix.values)
latent_items = svd.components_

reconstructed_scores = np.dot(latent_users, latent_items)
svd_reconstructed = pd.DataFrame(
    reconstructed_scores,
    index=interaction_matrix.index,
    columns=interaction_matrix.columns,
)

# Business-rule adjustment (age/wealth suitability)
svd_reconstructed = apply_business_rules(svd_reconstructed, all_clients, product_meta)

# Build top-3 recommendations per client
client_ids = (svd_reconstructed.index + 1).astype(int)
recommendations = []
for row_idx, client_id in zip(svd_reconstructed.index, client_ids):
    top_products = svd_reconstructed.loc[row_idx].sort_values(ascending=False).head(3)
    for product_id, score in top_products.items():
        recommendations.append(
            {
                "ClientID": int(client_id),
                "ProductID": int(product_id),
                "Score": float(score),
            }
        )

svd_recommendations = pd.DataFrame(recommendations).sort_values(
    ["ClientID", "Score"], ascending=[True, False]
)

print(f"Interaction matrix shape: {interaction_matrix.shape}")
print(f"Retained latent dimensions: {n_components}")

Interaction matrix shape: (5000, 11)
Retained latent dimensions: 6


In [17]:
svd_recommendations.head()

,ClientID,ProductID,Score
0,1,9,0.602292
1,1,6,0.584423
2,1,5,0.572173
3,2,10,0.369521
4,2,3,0.368983


### Validation


In [18]:
# Holdout validation for Truncated SVD
svd_val = TruncatedSVD(n_components=n_components, random_state=42)
latent_users_val = svd_val.fit_transform(interaction_train.values)
latent_items_val = svd_val.components_
svd_val_scores = pd.DataFrame(
    np.dot(latent_users_val, latent_items_val),
    index=interaction_train.index,
    columns=interaction_train.columns,
)
svd_val_scores = apply_business_rules(svd_val_scores, all_clients, product_meta)

svd_metrics = evaluate_recommender(
    svd_val_scores, interaction_matrix, holdout_pairs, k=3
)
pd.DataFrame([svd_metrics], index=["SVD Holdout Validation"])

,RMSE,Precision@3,Recall@3,NDCG@3
SVD Holdout Validation,0.36409,0.93028,0.93028,0.94944


### Generate recommendations per client


In [19]:
def recommend_for_client(client_id: int, top_n: int = 3) -> pd.DataFrame:
    client_rows = svd_recommendations[
        svd_recommendations["ClientID"] == int(client_id)
    ].copy()
    if client_rows.empty:
        return pd.DataFrame(columns=["ClientID", "ProductID", "Score", "Type", "Risk"])

    enriched = client_rows.merge(
        products_df[["IDProduct", "Type", "Risk"]],
        left_on="ProductID",
        right_on="IDProduct",
        how="left",
    ).drop(columns=["IDProduct"])

    return enriched.head(top_n)


sample_clients = (X_test.index[:5] + 1).astype(int).tolist()
sample_output = pd.concat(
    [recommend_for_client(cid, top_n=3) for cid in sample_clients], ignore_index=True
)
sample_output


,ClientID,ProductID,Score,Type,Risk
0,3880,9,0.512273,1,0.27
1,3880,6,0.498157,1,0.36
2,3880,5,0.487492,1,0.41
3,742,6,0.752010,1,0.36
4,742,9,0.745088,1,0.27
5,742,5,0.744685,1,0.41
6,3152,2,0.384250,0,0.30
7,3152,10,0.375298,0,0.13
8,3152,3,0.374050,0,0.12
9,4160,6,0.670763,1,0.36


## Autoencoder

An **autoencoder-based recommender system** learns a non‑linear, low‑dimensional representation (embedding) of each user from their interaction vector with items. By compressing and then reconstructing the user–item interaction matrix, the model captures complex, non‑additive patterns that linear methods (e.g., SVD) may overlook. In this project, the autoencoder serves as an **alternative latent factor model** for product recommendation.

Let $R \in \mathbb{R}^{m \times n}$ be the user–item interaction matrix, where $m$ is the number of clients and $n$ the number of products. For a given client $u$, the input is the row vector $\mathbf{r}_u \in \mathbb{R}^n$.

The autoencoder consists of two functions:

- **Encoder** $f_\theta : \mathbb{R}^n \to \mathbb{R}^k$ with $k \ll n$ (bottleneck dimension)
- **Decoder** $g_\phi : \mathbb{R}^k \to \mathbb{R}^n$

The forward pass for user $u$ is:

$$
\mathbf{z}_u = f_\theta(\mathbf{r}_u) = \sigma(W_e \mathbf{r}_u + \mathbf{b}_e)
$$

$$
\hat{\mathbf{r}}_u = g_\phi(\mathbf{z}_u) = \sigma'(W_d \mathbf{z}_u + \mathbf{b}_d)
$$

The model is trained by minimising the **reconstruction loss** over all users:

$$
\mathcal{L} = \frac{1}{m} \sum_{u=1}^m \| \mathbf{r}_u - \hat{\mathbf{r}}_u \|_2^2 + \lambda \|\theta, \phi\|_2^2
$$

Once trained, the reconstructed vector $\hat{\mathbf{r}}_u$ contains **predicted scores for all products**, including those the client has not previously interacted with. The top‑$K$ products with the highest predicted scores are selected as recommendations.


### Autoencoder Architecture


In [20]:
class RecommenderAutoencoder(nn.Module):
    def __init__(self, n_products, encoding_dim=10, dropout_rate=0.2):
        super().__init__()
        # Encoder: compress input vector to low-dimensional latent space
        self.encoder = nn.Sequential(
            nn.Linear(n_products, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Linear(32, encoding_dim),
        )
        # Decoder: reconstruct original vector from latent representation
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Linear(64, n_products),
            nn.Sigmoid(),  # Ensure output scores are in [0, 1]
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

### Training loop


In [21]:
# Convert interaction matrix to PyTorch tensor
X = torch.tensor(interaction_matrix.values, dtype=torch.float32)

model = RecommenderAutoencoder(n_products=X.shape[1], encoding_dim=8)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()

epochs = 100
batch_size = 64
dataset = torch.utils.data.TensorDataset(X)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in tqdm.tqdm(range(epochs), desc="Training Autoencoder"):
    epoch_loss = 0.0
    for batch in dataloader:
        inputs = batch[0]
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * inputs.size(0)
    avg_loss = epoch_loss / len(dataset)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.6f}")

Training Autoencoder:  21%|██        | 21/100 [00:03<00:11,  6.63it/s]

Epoch 20/100, Loss: 0.000992


Training Autoencoder:  41%|████      | 41/100 [00:06<00:08,  6.75it/s]

Epoch 40/100, Loss: 0.000930


Training Autoencoder:  61%|██████    | 61/100 [00:09<00:05,  6.62it/s]

Epoch 60/100, Loss: 0.000982


Training Autoencoder:  81%|████████  | 81/100 [00:12<00:02,  6.78it/s]

Epoch 80/100, Loss: 0.000792


Training Autoencoder: 100%|██████████| 100/100 [00:15<00:00,  6.66it/s]

Epoch 100/100, Loss: 0.000713


### Generating Recommendations


In [22]:
model.eval()
with torch.no_grad():
    reconstructed = model(X).numpy()

# For each client, retrieve top‑K product indices
k = 3
recommendations = []
for client_idx, row in enumerate(reconstructed):
    top_k_indices = row.argsort()[-k:][::-1]  # descending order
    for prod_idx in top_k_indices:
        recommendations.append(
            {
                "ClientID": interaction_matrix.index[client_idx],
                "ProductID": interaction_matrix.columns[prod_idx],
                "Score": row[prod_idx],
            }
        )
recommendations_df = pd.DataFrame(recommendations)

In [23]:
recommendations_df.head()

,ClientID,ProductID,Score
0,0,9,0.577717
1,0,6,0.573022
2,0,5,0.567140
3,1,10,0.372933
4,1,3,0.372264


In [24]:
# Holdout validation for Autoencoder
X_val_train = torch.tensor(interaction_train.values, dtype=torch.float32)

val_model = RecommenderAutoencoder(n_products=X_val_train.shape[1], encoding_dim=8)
val_optimizer = torch.optim.Adam(val_model.parameters(), lr=0.001, weight_decay=1e-5)
val_criterion = nn.MSELoss()

val_epochs = 60
val_batch_size = 64
val_dataset = torch.utils.data.TensorDataset(X_val_train)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=val_batch_size, shuffle=True
)

for epoch in tqdm.tqdm(range(val_epochs), desc="Validation Autoencoder"):
    val_epoch_loss = 0.0
    for batch in val_loader:
        val_inputs = batch[0]
        val_optimizer.zero_grad()
        val_outputs = val_model(val_inputs)
        val_loss = val_criterion(val_outputs, val_inputs)
        val_loss.backward()
        val_optimizer.step()
        val_epoch_loss += val_loss.item() * val_inputs.size(0)

    if (epoch + 1) % 20 == 0:
        avg_val_loss = val_epoch_loss / len(val_dataset)
        print(
            f"Validation model epoch {epoch + 1}/{val_epochs}, Loss: {avg_val_loss:.6f}"
        )

val_model.eval()
with torch.no_grad():
    val_reconstructed = val_model(X_val_train).numpy()

val_reconstructed_df = pd.DataFrame(
    val_reconstructed,
    index=interaction_train.index,
    columns=interaction_train.columns,
)

autoencoder_metrics = evaluate_recommender(
    val_reconstructed_df, interaction_matrix, holdout_pairs, k=3
)
pd.DataFrame([autoencoder_metrics], index=["Autoencoder Holdout Validation"])

Validation Autoencoder:  35%|███▌      | 21/60 [00:03<00:05,  6.63it/s]

Validation model epoch 20/60, Loss: 0.008275


Validation Autoencoder:  68%|██████▊   | 41/60 [00:06<00:02,  6.74it/s]

Validation model epoch 40/60, Loss: 0.005437


Validation Autoencoder: 100%|██████████| 60/60 [00:09<00:00,  6.60it/s]


Validation model epoch 60/60, Loss: 0.004658


,RMSE,Precision@3,Recall@3,NDCG@3
Autoencoder Holdout Validation,0.444168,0.934757,0.934757,0.928108
